In [3]:
import os
import warnings

import numpy as np
import pandas as pd

import librosa

from scipy import signal
from scipy.signal import find_peaks

warnings.filterwarnings("ignore")

In [4]:
os.listdir()

['.ipynb_checkpoints',
 'Group - 3 (2) (2).pdf',
 'holdout',
 'parkinson new data set edited',
 'parkinson.ipynb',
 'parkinsons.data',
 'parkinson_streamlit',
 'project_update',
 'project_update.zip',
 'random.ipynb',
 'results',
 'scr.zip',
 'Untitled.ipynb',
 'Validation',
 'Voice-Based-Parkinsons-Disease-Detection.pptx']

In [5]:
dataset_base = (
    r"C:\Users\tahmi\OneDrive\Desktop\dsp project"
    r"\parkinson new data set edited")

parkinson_folder = os.path.join(dataset_base, "Parkinson")

healthy_folder = os.path.join(dataset_base,"Healthy")

output_file = os.path.join(dataset_base,"parkinsons_uci_style_dataset_imporved.csv")

In [6]:
os.listdir(parkinson_folder)

['.ipynb_checkpoints',
 'Anna B',
 'Anna_B_features.csv',
 'Anna_B_UCI_style_features.csv',
 'Antonia G',
 'Daria L',
 'Domenico C',
 'Felicetta C',
 'Giovanni N',
 'Giulia L',
 'Giustina M',
 'Leonarda L',
 'Lucia R',
 'Luigi B',
 'Mario B',
 'Michele C',
 'Nicola M',
 'Nicola S',
 'Nicolò C',
 'Roberto L',
 'Roberto R',
 'Saverio S',
 'Ugo B',
 'Untitled.ipynb',
 'Vito L',
 'Vito S',
 'voice_features_python.csv']

In [7]:
print("=" * 70)
print("CHECKING DATASET PATHS")
print("=" * 70)

print("\nParkinson folder:")
print(parkinson_folder)

print("\nHealthy folder:")
print(healthy_folder)


if not os.path.isdir(parkinson_folder):

    raise ValueError(
        f"\nParkinson folder not found:\n"
        f"{parkinson_folder}"
    )


if not os.path.isdir(healthy_folder):

    raise ValueError(
        f"\nHealthy folder not found:\n"
        f"{healthy_folder}"
    )


print("\nBoth folders found successfully!")

CHECKING DATASET PATHS

Parkinson folder:
C:\Users\tahmi\OneDrive\Desktop\dsp project\parkinson new data set edited\Parkinson

Healthy folder:
C:\Users\tahmi\OneDrive\Desktop\dsp project\parkinson new data set edited\Healthy

Both folders found successfully!


In [8]:
feature_names = [

    "MDVP:Fo(Hz)",
    "MDVP:Fhi(Hz)",
    "MDVP:Flo(Hz)",

    "MDVP:Jitter(%)",
    "MDVP:Jitter(Abs)",
    "MDVP:RAP",
    "MDVP:PPQ",
    "Jitter:DDP",

    "MDVP:Shimmer",
    "MDVP:Shimmer(dB)",
    "Shimmer:APQ3",
    "Shimmer:APQ5",
    "MDVP:APQ",
    "Shimmer:DDA",

    "NHR",
    "HNR",

    "RPDE",
    "DFA",

    "spread1",
    "spread2",

    "D2",
    "PPE"
]

In [9]:
def get_wav_files(folder):

    wav_files = []

    for file in os.listdir(folder):

        if file.lower().endswith(".wav"):

            full_path = os.path.join(folder,file)

            wav_files.append(full_path)

    return sorted(wav_files)

In [10]:
def load_audio(filepath):

    y, sr = librosa.load(filepath,sr=None,mono=True)

    y = np.asarray( y,dtype=np.float64)

    # Remove DC component
    y = y - np.mean(y)

    # Remove linear trend
    y = signal.detrend(y)

    # Normalize amplitude
    peak = np.max(np.abs(y))

    if peak > 0:

        y = y / peak

    return y, sr

In [11]:
def load_audio(filepath):

    y, sr = librosa.load(filepath, sr=None, mono=True)

    y = np.asarray(y, dtype=np.float64)

    # Remove DC component
    y = y - np.mean(y)

    # Remove linear trend
    y = signal.detrend(y)

    # Normalize amplitude
    peak = np.max(np.abs(y))

    if peak > 0:
        y = y / peak

    # ============================================================
    # Wavelet Denoising
    # ============================================================
    import pywt

    wavelet = 'db6'

    # Determine suitable decomposition level
    max_level = pywt.dwt_max_level(len(y), pywt.Wavelet(wavelet).dec_len)
    level = min(5, max_level)

    # Wavelet decomposition
    coeffs = pywt.wavedec(y, wavelet, level=level)

    # Estimate noise from finest detail coefficients
    sigma = np.median(np.abs(coeffs[-1])) / 0.6745

    # Universal threshold
    threshold = sigma * np.sqrt(2 * np.log(len(y)))

    # Soft thresholding of detail coefficients
    denoised_coeffs = [coeffs[0]]

    for coeff in coeffs[1:]:
        denoised_coeffs.append(
            pywt.threshold(coeff, threshold, mode='soft')
        )

    # Reconstruct denoised signal
    y = pywt.waverec(denoised_coeffs, wavelet)

    # Keep exactly the original signal length
    y = y[:len(y)]  # <-- see note below

    # Re-normalize after denoising
    peak = np.max(np.abs(y))

    if peak > 0:
        y = y / peak

    return y, sr

In [12]:
def get_f0(y, sr):
    frame_length = int(0.040 * sr)
    
    hop_length = int( 0.010 * sr)

    if frame_length % 2 != 0:
        frame_length += 1

    if len(y) < frame_length:
        return np.array([])

    try:

        f0,_,__ = librosa.pyin(
            y,
            fmin=60,
            fmax=400,
            sr=sr,
            frame_length=frame_length,
            hop_length=hop_length

        )

        f0 = f0[ np.isfinite(f0)&(f0 > 0)]

        return f0

    except Exception:

        return np.array([])

In [13]:
def get_periods_amplitudes(y,sr,f0_mean):

    if not np.isfinite(f0_mean) or f0_mean <= 0:

        return np.array([]), np.array([])

    low = max(40,0.7 * f0_mean )

    high = min(1.4 * f0_mean,0.45 * sr)

    if high <= low:

        return np.array([]), np.array([])

    try:

        b, a = signal.butter(4, [ low / (sr / 2), high / (sr / 2)],btype="bandpass")

        yf = signal.filtfilt(b,a,y )

    except Exception:

        yf = y

    distance = max(2,int(0.7 * sr / f0_mean))

    peaks, _ = find_peaks(yf,distance=distance)

    if len(peaks) < 3:

        return np.array([]), np.array([])

    nominal_period = 1 / f0_mean

    periods = []
    amplitudes = []

    for i in range(len(peaks) - 1):

        period = (peaks[i + 1]-peaks[i]) / sr

        if (0.7 * nominal_period< period <1.4 * nominal_period):

            segment = y[peaks[i]: peaks[i + 1]]

            if len(segment) > 1:

                periods.append(period)

                amplitude = (np.max(segment) - np.min(segment))

                amplitudes.append(amplitude)

    return (
        np.array(periods),
        np.array(amplitudes)
    )

In [14]:
def calculate_jitter(periods):

    if len(periods) < 6:

        return (
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            np.nan
        )

    mean_period = np.mean(periods)

    if mean_period <= 0:

        return (
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            np.nan
        )

    jitter_abs = np.mean( np.abs( np.diff(periods) ) )

    jitter_percent = ( jitter_abs /mean_period ) * 100

    rap_values = []

    for i in range( 1,len(periods) - 1):

        local_mean = np.mean( periods[i - 1:i + 2] )

        rap_values.append(abs( periods[i] -local_mean))

    if len(rap_values) > 0:

        rap = (np.mean(rap_values)/ mean_period)

    else:

        rap = np.nan

    ppq_values = []

    for i in range(2,len(periods) - 2):

        local_mean = np.mean(periods[i - 2:i + 3])

        ppq_values.append(abs(periods[i]-local_mean))

    if len(ppq_values) > 0:

        ppq = (np.mean(ppq_values)/mean_period)

    else:

        ppq = np.nan

    if np.isfinite(rap):

        ddp = 3 * rap

    else:

        ddp = np.nan

    return (
        jitter_percent,
        jitter_abs,
        rap,
        ppq,
        ddp
    )

In [15]:
def calculate_shimmer(amplitudes):

    if len(amplitudes) < 6:

        return (
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            np.nan
        )

    mean_amplitude = np.mean(amplitudes)

    if mean_amplitude <= 0:

        return (
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            np.nan
        )

    shimmer = ( np.mean(np.abs( np.diff(amplitudes)))/mean_amplitude )

    shimmer_db_values = []

    for i in range( len(amplitudes) - 1):

        a1 = max(amplitudes[i],1e-12)

        a2 = max(amplitudes[i + 1],1e-12)

        value = abs(20 * np.log10( a2 / a1) )

        shimmer_db_values.append(value)

    shimmer_db = np.mean(shimmer_db_values)

    apq3_values = []

    for i in range(1,len(amplitudes) - 1):

        local_mean = np.mean(amplitudes[i - 1:i + 2] )

        apq3_values.append(abs(amplitudes[i]-local_mean ))

    if len(apq3_values) > 0:

        apq3 = (np.mean(apq3_values)/mean_amplitude )

    else:

        apq3 = np.nan

    apq5_values = []

    for i in range(2,len(amplitudes) - 2 ):

        local_mean = np.mean(amplitudes[i - 2:i + 3])

        apq5_values.append(abs(amplitudes[i]-local_mean) )

    if len(apq5_values) > 0:

        apq5 = (np.mean(apq5_values)/mean_amplitude)

    else:

        apq5 = np.nan

    apq_values = []

    if len(amplitudes) >= 11:

        for i in range(5,len(amplitudes) - 5):

            local_mean = np.mean(amplitudes[i - 5:i + 6])

            apq_values.append( abs(amplitudes[i]-local_mean))

    if len(apq_values) > 0:

        apq = (np.mean(apq_values)/mean_amplitude)

    else:

        apq = np.nan

    if np.isfinite(apq3):

        dda = 3 * apq3

    else:

        dda = np.nan

    return (
        shimmer,
        shimmer_db,
        apq3,
        apq5,
        apq,
        dda
    )

In [16]:
def calculate_hnr_nhr(y,sr,f0_mean):

    if not np.isfinite(f0_mean):

        return (
            np.nan,
            np.nan
        )

    if f0_mean <= 0:

        return (
            np.nan,
            np.nan
        )

    y = y - np.mean(y)

    autocorr = signal.correlate(y, y, mode="full")

    autocorr = autocorr[len(autocorr) // 2:]

    if autocorr[0] <= 0:

        return (
            np.nan,
            np.nan
        )

    autocorr = ( autocorr/autocorr[0])

    pitch_period = int( round(sr / f0_mean))

    low_lag = max( 1, int(0.8 * pitch_period))

    high_lag = min(len(autocorr) - 1,int(1.2 * pitch_period))

    if high_lag <= low_lag:

        return (
            np.nan,
            np.nan
        )

    r = np.max(autocorr[low_lag:high_lag + 1])

    r = np.clip(r,1e-6,1 - 1e-6)

    hnr = ( 10*np.log10(r / (1 - r)))

    nhr = ( (1 - r) /r)

    return (nhr,hnr)

In [17]:
def calculate_rpde(periods):

    if len(periods) < 20:

        return np.nan

    std_value = np.std(periods)

    if std_value == 0:

        return np.nan

    x = (periods-np.mean(periods)) / std_value

    hist, _ = np.histogram( x,bins=20)

    hist = hist[hist > 0]

    if len(hist) < 2:

        return np.nan

    probabilities = (hist/np.sum(hist))

    entropy = (-np.sum(probabilities*np.log(probabilities) ))

    rpde = (entropy/np.log(len(probabilities)) )

    return rpde

In [18]:
def calculate_dfa(x):

    x = np.asarray(x)

    x = x[np.isfinite(x)]

    if len(x) < 30:

        return np.nan

    y = np.cumsum(x-np.mean(x))

    max_scale = len(y) // 4

    if max_scale < 8:

        return np.nan

    scales = np.unique(np.logspace( np.log10(4),np.log10(max_scale),12).astype(int))

    fluctuation = []

    valid_scales = []

    for scale_value in scales:

        number_segments = (len(y)//scale_value)

        if number_segments < 2:

            continue

        rms_values = []

        for j in range(number_segments):

            segment = y[j * scale_value:(j + 1) * scale_value]

            t = np.arange(scale_value)

            coefficients = np.polyfit(t,segment,1)

            trend = np.polyval( coefficients,t)

            rms = np.sqrt(np.mean( (segment- trend)**2 ))

            rms_values.append(rms)

        F = np.sqrt(np.mean( np.array(rms_values)**2))

        if F > 0:

            fluctuation.append(F)

            valid_scales.append(scale_value)

    if len(valid_scales) < 3:

        return np.nan

    alpha = np.polyfit(np.log(valid_scales),np.log(fluctuation),1)[0]

    return alpha

In [19]:
def calculate_spread(f0):

    if len(f0) < 10:

        return (
            np.nan,
            np.nan
        )

    x = np.log(f0)

    spread1 = np.std(x)

    spread2 = np.std(np.diff(x))

    return (
        spread1,
        spread2
    )

In [20]:
def calculate_d2(x):

    if len(x) < 50:

        return np.nan

    std_value = np.std(x)

    if std_value == 0:

        return np.nan

    x = (x-np.mean(x)) / std_value

    if len(x) > 500:
        x = x[:500]

    points = np.column_stack( ( x[:-1],x[1:]))

    distances = []

    for i in range(len(points)):

        d = np.linalg.norm(points[i + 1:] -points[i],axis=1)

        d = d[ d > 0 ]

        distances.extend(d)

    distances = np.array(distances)

    if len(distances) < 20:

        return np.nan

    radii = np.percentile(distances,[10, 20, 30, 40])

    correlation = []

    for radius in radii:

        value = np.mean(distances<radius)

        correlation.append(value)

    correlation = np.array(correlation)

    valid = ((correlation > 0) & (correlation < 1))

    if np.sum(valid) < 2:

        return np.nan

    d2 = np.polyfit( np.log(radii[valid]),np.log(correlation[valid]),1)[0]

    return d2

In [21]:
def calculate_ppe(f0):

    if len(f0) < 20:

        return np.nan

    x = np.log(f0)

    x = (x-np.median(x))

    hist, _ = np.histogram( x, bins=20 )

    hist = hist[hist > 0]

    if len(hist) == 0:

        return np.nan

    probabilities = (hist / np.sum(hist))

    ppe = (-np.sum(probabilities*np.log(probabilities)) )

    return ppe

In [22]:
def extract_22_features(filepath):

    # Load audio
    y, sr = load_audio(filepath)

    # Extract F0
    f0 = get_f0(y, sr)

    # If F0 extraction fails
    if len(f0) < 5:

        return {
            feature: np.nan
            for feature in feature_names
        }

    # Pitch features
    fo = np.mean(f0)
    fhi = np.max(f0)
    flo = np.min(f0)

    # Periods and amplitudes
    periods, amplitudes = get_periods_amplitudes( y, sr,fo)

    # Jitter
    (jitter_percent,jitter_abs,rap,ppq,ddp) = calculate_jitter(periods)

    # Shimmer
    (shimmer,shimmer_db,apq3,apq5,apq,dda) = calculate_shimmer(amplitudes)

    # Noise features
    nhr, hnr = calculate_hnr_nhr( y,sr,fo)

    # Nonlinear features
    rpde = calculate_rpde(periods)

    dfa = calculate_dfa(f0)

    spread1, spread2 = calculate_spread(f0)

    d2 = calculate_d2(f0)

    ppe = calculate_ppe(f0)

    # Return all features
    features = {

        "MDVP:Fo(Hz)": fo,
        "MDVP:Fhi(Hz)": fhi,
        "MDVP:Flo(Hz)": flo,

        "MDVP:Jitter(%)": jitter_percent,
        "MDVP:Jitter(Abs)": jitter_abs,
        "MDVP:RAP": rap,
        "MDVP:PPQ": ppq,
        "Jitter:DDP": ddp,

        "MDVP:Shimmer": shimmer,
        "MDVP:Shimmer(dB)": shimmer_db,
        "Shimmer:APQ3": apq3,
        "Shimmer:APQ5": apq5,
        "MDVP:APQ": apq,
        "Shimmer:DDA": dda,

        "NHR": nhr,
        "HNR": hnr,

        "RPDE": rpde,
        "DFA": dfa,

        "spread1": spread1,
        "spread2": spread2,

        "D2": d2,
        "PPE": ppe
    }

    return features

In [23]:
def process_group(group_folder,status,group_name):

    results = []

    subjects = sorted(folder for folder in os.listdir(group_folder) if os.path.isdir(os.path.join(group_folder, folder)))

    

    print("\n" + "=" * 70)

    print(f"PROCESSING {group_name.upper()}")

    print(f"Number of subjects: {len(subjects)}")

    print("=" * 70)


    for subject_number, subject in enumerate(subjects,start=1):

        subject_folder = os.path.join( group_folder,subject)

        wav_files = get_wav_files(subject_folder)

        print(
            f"\n[{subject_number}/{len(subjects)}] "
            f"{subject}")

        print(f"    WAV files found: "
            f"{len(wav_files)}"
        )


        for file_number, filepath in enumerate( wav_files,start=1):

            filename = os.path.basename(filepath )

            print(f"    [{file_number}/{len(wav_files)}] "
                f"{filename}",

                end=" ... "

            )

            try:

                features = extract_22_features(filepath)

                row = {}

                row["name"] = filename

                row["subject"] = subject

                row["status"] = status


                for feature in feature_names:

                    row[feature] = features[feature]

                results.append(row)

                print("DONE")


            except Exception as e:

                print(
                    f"ERROR: {e}"
                )

    return results

In [24]:
parkinson_results = process_group(group_folder=parkinson_folder,status=1,group_name="Parkinson")
healthy_results = process_group( group_folder=healthy_folder, status=0,group_name="Healthy")
all_results = (parkinson_results + healthy_results)


PROCESSING PARKINSON
Number of subjects: 23

[1/23] .ipynb_checkpoints
    WAV files found: 0

[2/23] Anna B
    WAV files found: 12
DONE[1/12] D1ABNINSAC46F240120171756.wav ... 
DONE[2/12] D2ABNINSAC46F240120171756.wav ... 
DONE[3/12] VA1ABNINSAC46F240120171758.wav ... 
DONE[4/12] VA2ABNINSAC46F240120171759.wav ... 
DONE[5/12] VE1ABNINSAC46F240120171800.wav ... 
DONE[6/12] VE2ABNINSAC46F240120171801.wav ... 
DONE[7/12] VI1ABNINSAC46F240120171801.wav ... 
DONE[8/12] VI2ABNINSAC46F240120171802.wav ... 
DONE[9/12] VO1ABNINSAC46F240120171802.wav ... 
DONE[10/12] VO2ABNINSAC46F240120171803.wav ... 
DONE[11/12] VU1ABNINSAC46F240120171804.wav ... 
DONE[12/12] VU2ABNINSAC46F240120171804.wav ... 

[3/23] Antonia G
    WAV files found: 12
DONE[1/12] D1AGNUTGOL52F100220171046.wav ... 
DONE[2/12] D2AGNUTGOL52F100220171046.wav ... 
DONE[3/12] VA1AGNUTGOL52F100220171049.wav ... 
DONE[4/12] VA2AGNUTGOL52F100220171049.wav ... 
DONE[5/12] VE1AGNUTGOL52F100220171050.wav ... 
DONE[6/12] VE2AGNUTGOL52F1

In [25]:
columns = (["name", "subject"]+feature_names+["status"])


df = pd.DataFrame(all_results,columns=columns)

In [26]:
df.to_csv( output_file,index=False)

In [27]:
print("\n")

print("=" * 70)

print("DATASET CREATION COMPLETE")

print("=" * 70)


print( f"\nTotal recordings: "f"{len(df)}")


print(f"Total columns: "f"{len(df.columns)}")


print(
    f"\nParkinson recordings: "
    f"{(df['status'] == 1).sum()}"
)


print(
    f"Healthy recordings: "
    f"{(df['status'] == 0).sum()}"
)


print(
    f"\nParkinson subjects: "
    f"{df[df['status'] == 1]['subject'].nunique()}"
)


print(
    f"Healthy subjects: "
    f"{df[df['status'] == 0]['subject'].nunique()}"
)


print(
    f"\nDataset saved to:"
)


print(
    output_file
)



DATASET CREATION COMPLETE

Total recordings: 668
Total columns: 25

Parkinson recordings: 319
Healthy recordings: 349

Parkinson subjects: 22
Healthy subjects: 22

Dataset saved to:
C:\Users\tahmi\OneDrive\Desktop\dsp project\parkinson new data set edited\parkinsons_uci_style_dataset_imporved.csv


In [1]:
pip install PyWavelets

     ---------------------------------------- 0.0/4.2 MB ? eta -:--:--
     ---------------------------------------- 0.0/4.2 MB ? eta -:--:--
      --------------------------------------- 0.1/4.2 MB 1.7 MB/s eta 0:00:03
      --------------------------------------- 0.1/4.2 MB 1.7 MB/s eta 0:00:03
     - -------------------------------------- 0.2/4.2 MB 1.2 MB/s eta 0:00:04
     -- ------------------------------------- 0.2/4.2 MB 1.1 MB/s eta 0:00:04
     -- ------------------------------------- 0.3/4.2 MB 1.3 MB/s eta 0:00:04
     --- ------------------------------------ 0.3/4.2 MB 1.2 MB/s eta 0:00:04
     --- ------------------------------------ 0.4/4.2 MB 1.2 MB/s eta 0:00:04
     --- ------------------------------------ 0.4/4.2 MB 1.1 MB/s eta 0:00:04
     ---- ----------------------------------- 0.5/4.2 MB 1.1 MB/s eta 0:00:04
     ---- ----------------------------------- 0.5/4.2 MB 1.0 MB/s eta 0:00:04
     ----- ---------------------------------- 0.6/4.2 MB 1.0 MB/s eta 0:00:04



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: C:\Users\tahmi\ml_env\Scripts\python.exe -m pip install --upgrade pip
